# Ingesta bronze - products
El archivo tiene un `;` al final de cada linea y algunos nombres traen comillas escapadas.
Por eso se agrega una columna auxiliar al schema (luego se elimina) y se configuran `quote` y `escape`.

In [ ]:
%run ./00_utils

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

params = obtener_parametros()
CATALOGO, STORAGE = params["catalogo"], params["storage"]
ENTIDAD = "products"
TABLA_DESTINO = f"{CATALOGO}.bronze.{ENTIDAD}"

In [ ]:
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True),
    StructField("_columna_vacia", StringType(), True),
])

In [ ]:
try:
    ruta = ruta_raw(STORAGE, ENTIDAD)
    log(f"Leyendo {ruta}", entidad=ENTIDAD)

    df_raw = (spark.read
              .option("header", True)
              .option("sep", ";")
              .option("quote", '"')
              .option("escape", '"')
              .option("encoding", "UTF-8")
              .schema(products_schema)
              .csv(ruta))

    df_bronze = agregar_auditoria(df_raw).drop("_columna_vacia")
    validar_no_vacio(df_bronze, ENTIDAD)
    escribir_tabla(df_bronze, TABLA_DESTINO, modo="overwrite")

except Exception as e:
    log(f"Error en la ingesta: {e}", "ERROR", ENTIDAD)
    raise

In [ ]:
display(spark.table(TABLA_DESTINO).limit(10))